# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5-mini"

/Users/tayjiasheng/AI Projects/llm_engineering/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:41<00:00, 13.94s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: Razer and More Gaming Deals at Woot: Up to 64% off + $5 off + free shipping w/ Prime\nDetails: Get deals on a selection of Razer gaming accessories in this sale at Woot. Even better, you can use promo code "RAZERWOOT" to get an extra $5 off your order. (If it\'s your first time shopping at Woot, the coupon will net you $10 off your order.) We\'ve pictured the Razer Huntsman V2 Analog Gaming Keyboard for $64.99 (72% off) after coupon. Sale ends June 7. Shop Now at Woot! An Amazon Company\nFeatures: \nURL: https://www.dealnews.com/Razer-and-More-Gaming-Deals-at-Woot-Up-to-64-off-5-off-free-shipping-w-Prime/21836605.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price


This prompt requires experimentation!!! it's not a one-off thing.


In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals


def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += "\n\n".join(
        [scrape.describe() for scrape in scraped]
    )  # we are including the 30 things that we scraped.
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Certified Refurb Bose SoundLink Flex Portable Bluetooth Speaker + free shipping w/ Prime
Details: The Bose SoundLink Flex Gen 1 is $69.99 in certified refurbished condition at Woot, backed by a 1-year Bose manufacturer limited warranty. It's within a buck of the best-ever price we've seen for this model. The speaker carries an IP67 rating for waterproofing and dustproofing and 

In [9]:
DealSelection.model_json_schema()
# DealSelection is a class, which is a subclass of the BaseModel (from pydantic), that's why it has access to the function, model_json_schema()
# this json schema is what we are providing to the openai model

{'$defs': {'Deal': {'description': 'A class to Represent a Deal with a summary description',
   'properties': {'product_description': {'description': "Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose.",
     'title': 'Product Description',
     'type': 'string'},
    'price': {'description': 'The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200',
     'title': 'Price',
     'type': 'number'},
    'url': {'description': 'The URL of the deal, as provided in the input',
     'title': 'Url',
     'type': 'string'}},
   'required': ['product_description', 'price', 'url'],
   'title': 'Deal',
   'type': 'object'}},
 'description': 'A clas

In [10]:
response = openai.chat.completions.parse(
    model=MODEL,
    messages=messages,
    response_format=DealSelection,  # this one is in deals.py, DealSelection is a subclass of BaseModel
    reasoning_effort="minimal",
)  # .parse() is where we have the STRUCTURED OUTPUTS, instead of chat.completions.create() (can refer to week 5)
results = response.choices[0].message.parsed
# note: it is .parsed, not .content like what we are used to
results  # the output is a pydantic object (which is converted from the json schema that the LLM outputs)

DealSelection(deals=[Deal(product_description='The Bose SoundLink Flex Gen 1 is a portable Bluetooth speaker with an IP67 rating for dust and waterproof protection, making it suitable for outdoor use. It delivers up to 12 hours of battery life per charge and carries Bose audio tuning in a compact rugged enclosure. This certified refurbished unit is backed by a one-year Bose manufacturer limited warranty.', price=69.99, url='https://www.dealnews.com/Certified-Refurb-Bose-Sound-Link-Flex-Portable-Bluetooth-Speaker-free-shipping-w-Prime/21836604.html?iref=rss-c142'), Deal(product_description='The Samsung Galaxy A54 5G is an unlocked Android smartphone featuring a 6.4-inch FHD+ Super AMOLED display, 128GB internal storage, a triple-lens rear camera system with optical image stabilization, and a 5,000mAh battery with fast charging support. It includes Gorilla Glass 5 for display protection and 5G connectivity for faster mobile data.', price=233.0, url='https://www.dealnews.com/Unlocked-Sams

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The Bose SoundLink Flex Gen 1 is a portable Bluetooth speaker with an IP67 rating for dust and waterproof protection, making it suitable for outdoor use. It delivers up to 12 hours of battery life per charge and carries Bose audio tuning in a compact rugged enclosure. This certified refurbished unit is backed by a one-year Bose manufacturer limited warranty.
69.99
https://www.dealnews.com/Certified-Refurb-Bose-Sound-Link-Flex-Portable-Bluetooth-Speaker-free-shipping-w-Prime/21836604.html?iref=rss-c142

The Samsung Galaxy A54 5G is an unlocked Android smartphone featuring a 6.4-inch FHD+ Super AMOLED display, 128GB internal storage, a triple-lens rear camera system with optical image stabilization, and a 5,000mAh battery with fast charging support. It includes Gorilla Glass 5 for display protection and 5G connectivity for faster mobile data.
233.0
https://www.dealnews.com/Unlocked-Samsung-Galaxy-A54-5-G-128-GB-Android-Phone-free-shipping/21836530.html?iref=rss-c142

The Vizio VQD50R-10 

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

A quick note - the cost of turning unstructured inputs to structured outputs with LLMs is:

1. Latency
2. Unpredicatability occassionally from LLMs, but nonetheless they are remarkable.

This has many applications!


In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [15]:
result

DealSelection(deals=[Deal(product_description='The Bose SoundLink Flex Gen 1 is a portable Bluetooth speaker designed for outdoor use with an IP67 rating for waterproofing and dustproofing. It offers balanced audio in a compact, rugged chassis and provides up to 12 hours of battery life per charge. The unit is a certified refurbished model backed by a one-year Bose manufacturer limited warranty.', price=69.99, url='https://www.dealnews.com/Certified-Refurb-Bose-Sound-Link-Flex-Portable-Bluetooth-Speaker-free-shipping-w-Prime/21836604.html?iref=rss-c142'), Deal(product_description='The Samsung Galaxy A54 5G is an unlocked Android smartphone featuring a 6.4" FHD+ Super AMOLED display, 128GB of internal storage, a triple-lens rear camera system with optical image stabilization, and a 5,000mAh battery with Super Fast Charging. It supports 5G connectivity and is built with Gorilla Glass 5 for added durability.', price=233.0, url='https://www.dealnews.com/Unlocked-Samsung-Galaxy-A54-5-G-128-

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER*USER=\_put the key that's on the top right of your Pushover home screen and probably starts with a u*  
PUSHOVER*TOKEN=\_put the key when you click into your new application called Agents (or whatever) and probably starts with an a*

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.


In [16]:
load_dotenv(override=True)

True

In [3]:
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_url = "https://api.pushover.net/1/messages.json"
import telebot

chat_id = os.getenv("CHAT_ID")
bot_token = os.getenv("BOT_TOKEN")
bot = telebot.TeleBot(token=bot_token)

In [18]:
# if pushover_user:
#     print(f"Pushover user found and starts with {pushover_user[0]}")
# else:
#     print("Pushover user not found")

# if pushover_token:
#     print(f"Pushover token found and starts with {pushover_token[0]}")
# else:
#     print("Pushover token not found")

In [19]:
# def push(message):
#     print(f"Push: {message}")
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     requests.post(pushover_url, data=payload)

In [20]:
# push("MASSIVE DEAL!!")
bot.send_message(chat_id, "MASSIVE DEAL!!")

In [2]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [4]:
agent.notify(
    "A special deal on Sumsung 60 inch LED TV going at a great bargain",
    300,
    1000,
    "www.samsung.com",
)